# AE+Ridge GPU Backtest on Colab

> **Deprecated:** Use `dl_runner.ipynb` instead. It supports both PatchTSMixer
> and AE+Ridge with MCP-driven cell-by-cell execution and Drive-persisted
> status tracking. Set `EXPERIMENT = "ae_ridge"` in Cell 2.

Run an AE+Ridge GPU backtest on Google Colab.

**Requirements:** Colab GPU runtime (T4 or better). Go to *Runtime > Change runtime type > GPU*.

In [ ]:
# --- 1. Setup: clone repo and install deps ---
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"  # FILL IN: your repo URL
REPO_DIR = "harxhar"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q torch transformers numpy pandas scikit-learn tqdm pyarrow

In [ ]:
# --- 2. Verify GPU and configure CUDA ---
from projects.dl.notebook_utils import configure_cuda, verify_gpu

verify_gpu()
configure_cuda()

## Data Configuration

In [ ]:
# --- 4. Upload your data ---
# Option A: Upload from local machine
# from google.colab import files
# uploaded = files.upload()  # upload your parquet files to all30min/

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/harxhar_data/all30min'

# Option C: Data already in repo
DATA_PATH = "all30min"
RESULTS_DIR = "results_ae_ridge"

## AE+Ridge Backtest

In [ ]:
import copy

import numpy as np
from core.data import load_and_prep_data_strided
from core.evaluation.metrics import calculate_global_metrics

from projects.dl.backtest.gpu_engine import run_ae_multigpu_backtest
from projects.dl.config import AE_RIDGE_GPU_CONFIG

np.random.seed(42)

ae_config = copy.deepcopy(AE_RIDGE_GPU_CONFIG)
ae_config["data_path"] = DATA_PATH
ae_config["gpu_count"] = 1  # Colab has 1 GPU

ae_hparams = {
    "exog_cols": "none",
    "use_transform_exog": False,
    "use_diurnal": True,
    "allow_missing": False,
    "use_winsor": True,
    "feature_type": "raw",
    "lag_scope": "global",
}

X_np, y_np, dates, baselines, features = load_and_prep_data_strided(ae_hparams, ae_config["data_path"])
ae_config["model"]["n_features"] = X_np.shape[1]
print(f"Data: {X_np.shape[0]:,} samples, {X_np.shape[1]} features")

In [ ]:
from projects.dl.notebook_utils import print_metrics

ae_results = run_ae_multigpu_backtest(X_np, y_np, dates, baselines, ae_config)

print_metrics(calculate_global_metrics(ae_results))

In [ ]:
from projects.dl.notebook_utils import save_results

csv_path = save_results(ae_results, RESULTS_DIR, "ae_ridge_results.csv")

In [ ]:
from projects.dl.notebook_utils import download_results

download_results(csv_path)